# MemPrimitive 功能展示

## 1. MemPrimitive方法的pipeline

在 `MemPrimitive` 里，memory system 不是黑盒方法名，而是一条可以逐段替换的流水线：

```text
ingest:
  unit_formation
  -> representation
  -> write_trigger
  -> organization
  -> evolution_trigger
  -> memory_evolution

recall:
  retrieval
  -> readout
```

下面的单元格会依次展示：

- 最小可用 memory module
- 多层 topology
- 可组合 trigger
- dispatch fan-out
- 适合继续运行的 demonstration 脚本

In [12]:
from pathlib import Path
import sys
from pprint import pprint

repo_root = Path.cwd()
if not (repo_root / "memprimitive").exists():
    for candidate in [repo_root, *repo_root.parents]:
        if (candidate / "memprimitive").exists():
            repo_root = candidate
            break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"repo_root = {repo_root}")

repo_root = d:\Git\MemPrimitive-MemEngineDemo\MemPrimitive


In [13]:
from memprimitive import (
    DispatchOrganization,
    IncompatibleCompositionError,
    MemoryPipeline,
    MemoryStore,
    Observation,
    Packet,
    Query,
    StoreLayerSpec,
    StoreTopology,
)
from memprimitive.baselines import (
    AlwaysWriteTrigger,
    AppendOnlyEvolution,
    AppendOrganization,
    BasicRepresentation,
    ConcatenateReadout,
    EmbeddingSimilarityRetrieval,
    EntityRetrieval,
    GraphLinkEvolution,
    GraphNeighborAppendEvolution,
    GraphNeighborContextTraceEvolution,
    GraphAppendOrganization,
    GraphReadout,
    GraphSeedAndExpandRetrieval,
    LayerAwareRetrieval,
    NeighborExistsEvolutionTrigger,
    PassThroughUnitFormation,
    KeywordRepresentation,
    RecencyRetrieval,
    ThresholdEvolutionTrigger,
)
from memprimitive.baselines._trigger_family import (
    AlwaysOpenGate,
    BooleanGatePolicy,
    ConstantSignal,
    ThresholdPolicy,
    WeightedSumScorer,
)
from memprimitive.baselines.evolution_trigger import compose_evolution_trigger
from memprimitive.baselines.write_trigger import compose_write_trigger

## 2. 最小可用 memory module

先拼出一个最小闭环，再逐个替换 slot。

In [14]:
minimal_pipeline = MemoryPipeline(
    unit_formation=PassThroughUnitFormation(),
    representation=BasicRepresentation(),
    write_trigger=AlwaysWriteTrigger(),
    organization=AppendOrganization(),
    retrieval=RecencyRetrieval(top_k=2),
    readout=ConcatenateReadout(),
)

minimal_pipeline

In [15]:
minimal_pipeline.ingest(Observation(text="The user likes concise examples.", source="dialogue"))
minimal_pipeline.ingest(Observation(text="The user works on compositional memory.", source="notes"))

minimal_readout = minimal_pipeline.recall(Query(text="What does the user like?"))
print(minimal_readout.text)
print("source_ids:", minimal_readout.source_ids)

The user works on compositional memory.
The user likes concise examples.
source_ids: ['rec-2', 'rec-1']


可以在上面的代码里直接替换某个 slot，比如：

- 把 `AlwaysWriteTrigger()` 换成组合 trigger
- 把 `RecencyRetrieval(top_k=2)` 换成别的 retrieval
- 把 `AppendOrganization()` 换成分层路由或 graph organization

这就是 `MemPrimitive` 灵活性的核心来源。

## 3. topology：结构是声明出来的

多层 memory 不是另起一套系统，而是 `StoreTopology` 的一部分。

In [16]:
topology = StoreTopology.from_layers(
    [
        StoreLayerSpec(name="working", theme="working", indices=("temporal", "keyword")),
        StoreLayerSpec(name="episodic", theme="session_memory", indices=("temporal", "keyword")),
        StoreLayerSpec(
            name="knowledge_graph",
            theme="knowledge_graph",
            shape="Graph",
            indices=("graph", "entity"),
        ),
    ]
)
store = MemoryStore(topology=topology)

topology_pipeline = MemoryPipeline(
    unit_formation=PassThroughUnitFormation(),
    representation=BasicRepresentation(),
    write_trigger=AlwaysWriteTrigger(),
    organization=AppendOrganization(target_layer="episodic"),
    retrieval=RecencyRetrieval(top_k=2, layer="episodic"),
    readout=ConcatenateReadout(),
    store=store,
)

packet = topology_pipeline.ingest(
    Observation(text="The user wants a store with an explicit topology.", source="notes")
)
topology_pipeline.ingest(
    Observation(text="The episodic layer keeps recent dialogue-like memories.", source="notes")
)

pprint([
    {
        "name": layer.name,
        "theme": layer.theme,
        "shape": layer.shape,
        "indices": layer.indices,
    }
    for layer in topology_pipeline.store.topology.layers
])
print("organization target layer:", packet.trace["organization"]["target_layer"])
print("records per layer:", {name: topology_pipeline.store.count(name) for name in topology_pipeline.store.topology.layer_names})

[{'indices': ('temporal', 'keyword'),
  'name': 'working',
  'shape': 'Flat',
  'theme': 'working'},
 {'indices': ('temporal', 'keyword'),
  'name': 'episodic',
  'shape': 'Flat',
  'theme': 'session_memory'},
 {'indices': ('graph', 'entity'),
  'name': 'knowledge_graph',
  'shape': 'Graph',
  'theme': 'knowledge_graph'}]
organization target layer: episodic
records per layer: {'working': 0, 'episodic': 2, 'knowledge_graph': 0}


可以修改：

- 增减 layer
- 改 `shape`
- 改 `indices`
- 改 `organization` 和 `retrieval` 指向的 layer

结构层和行为层是解耦的。

## 4. trigger可以直接组装

其触发逻辑可以拆成 signal、scorer、gate、policy 四段。

In [17]:
trigger_pipeline = MemoryPipeline(
    unit_formation=PassThroughUnitFormation(),
    representation=BasicRepresentation(),
    write_trigger=compose_write_trigger(
        name="demo_threshold_write_trigger",
        signal_providers=(ConstantSignal(signal_name="importance_hint", value=0.8),),
        scorer=WeightedSumScorer(weights={"importance_hint": 1.0}),
        gate=AlwaysOpenGate(),
        policy=ThresholdPolicy(threshold=0.5),
    ),
    organization=AppendOrganization(),
    evolution_trigger=compose_evolution_trigger(
        name="demo_boolean_evolution_trigger",
        signal_providers=(ConstantSignal(signal_name="after_write_ready", value=1.0),),
        scorer=WeightedSumScorer(weights={"after_write_ready": 1.0}),
        gate=AlwaysOpenGate(),
        policy=BooleanGatePolicy(),
    ),
    memory_evolution=AppendOnlyEvolution(),
    retrieval=RecencyRetrieval(top_k=2),
    readout=ConcatenateReadout(),
)

trigger_packet = trigger_pipeline.ingest(
    Observation(text="The user is exploring compositional triggers.", source="notes")
)

print("write_trigger trace:")
pprint(trigger_packet.trace["write_trigger"])
print()
print("evolution_trigger trace:")
pprint(trigger_packet.trace["evolution_trigger"])


write_trigger trace:
{'decisions': [True],
 'family': 'stage1_trigger_family',
 'gate': 'always_open',
 'module': 'demo_threshold_write_trigger',
 'output_field': 'decisions',
 'per_unit': [{'decision': True,
               'gate': True,
               'score': 0.8,
               'signals': {'importance_hint': 0.8},
               'unit_id': 'unit-66447779309d4ad599e5c95386a6b50e'}],
 'policy': 'threshold',
 'scorer': 'weighted_sum'}

evolution_trigger trace:
{'evolution_decisions': [True],
 'family': 'stage1_trigger_family',
 'gate': 'always_open',
 'module': 'demo_boolean_evolution_trigger',
 'output_field': 'evolution_decisions',
 'per_unit': [{'decision': True,
               'gate': True,
               'score': 1.0,
               'signals': {'after_write_ready': 1.0},
               'unit_id': 'unit-66447779309d4ad599e5c95386a6b50e'}],
 'policy': 'boolean_gate',
 'scorer': 'weighted_sum'}


可以直接修改：

- `ConstantSignal` 的值
- `WeightedSumScorer` 的权重
- `ThresholdPolicy` 的阈值
- `BooleanGatePolicy()` 换成别的 policy

## 5. dispatch：一次 ingest 进入多个 memory view

如果想让同一份输入同时进入普通层和 graph 层，可以用 `DispatchOrganization`。

In [18]:
dispatch_topology = StoreTopology.from_layers(
    [
        StoreLayerSpec(name="working", theme="working", indices=("temporal", "keyword")),
        StoreLayerSpec(
            name="knowledge_graph",
            theme="knowledge_graph",
            shape="Graph",
            indices=("graph", "entity"),
        ),
    ]
)
dispatch_store = MemoryStore(topology=dispatch_topology)

dispatch_pipeline = MemoryPipeline(
    representation=BasicRepresentation(elements=("text", "tags", "entities", "triple", "tags")),
    organization=DispatchOrganization(
        (
            AppendOrganization(target_layer="working"),
            GraphAppendOrganization(target_layer="knowledge_graph"),
        ),
        primary_index=0,
    ),
    retrieval=LayerAwareRetrieval(
        default_retriever=RecencyRetrieval(top_k=2),
        retriever_by_layer={"knowledge_graph": EntityRetrieval(top_k=2)},
        top_k=4,
    ),
    readout=ConcatenateReadout(separator="\n\n"),
    store=dispatch_store,
)

dispatch_pipeline.ingest(Observation(text="Alice is debugging the retrieval merge order.", source="dialogue"))
dispatch_pipeline.ingest(Observation(text="The current task is to explain graph-backed recall.", source="dialogue"))

dispatch_readout = dispatch_pipeline.recall(Query(text="Alice"))
retrieval_packet, _ = dispatch_pipeline.retrieval.run(Packet(query=Query(text="Alice")), dispatch_store)

print("records per layer:")
pprint({name: dispatch_store.count(name) for name in dispatch_store.topology.layer_names})
print()
print("layer-aware retrieval trace:")
pprint(retrieval_packet.trace["retrieval"])
print()
print(dispatch_readout.text)
print("source_ids:", dispatch_readout.source_ids)

records per layer:
{'knowledge_graph': 2, 'working': 2}

layer-aware retrieval trace:
{'active_layers': ['working', 'knowledge_graph'],
 'final_returned_count': 2,
 'merge_strategy': 'global_rank',
 'module': 'layer_aware_retrieval',
 'per_layer': [{'candidate_count': 2,
                'layer': 'working',
                'module': 'recency_retrieval',
                'returned_count': 1,
                'trace': {'candidate_count': 2,
                          'matched_by_keyword': True,
                          'module': 'recency_retrieval',
                          'top_k': 2}},
               {'candidate_count': 2,
                'layer': 'knowledge_graph',
                'module': 'entity_retrieval',
                'returned_count': 1,
                'trace': {'candidate_count': 2,
                          'module': 'entity_retrieval',
                          'top_k': 2}}],
 'total_merged_count': 2}

Alice is debugging the retrieval merge order.

Alice is debugging the re

- 同一输入可以同时进入多个 memory view
- 多层结构和 retrieval 编排仍然保持统一接口
- graph 能力不是外挂，而是 pipeline 里的正常组合方式

## 6. 复现一个 RAG baseline

展示一个最经典、最容易理解的 baseline：

- 写入时生成 embedding
- 查询时用 embedding similarity 检索
- 最后把召回结果拼成上下文

In [19]:
rag_pipeline = MemoryPipeline(
    unit_formation=PassThroughUnitFormation(),
    representation=BasicRepresentation(elements=("text", "embedding")),
    write_trigger=AlwaysWriteTrigger(),
    organization=AppendOrganization(),
    retrieval=EmbeddingSimilarityRetrieval(top_k=2),
    readout=ConcatenateReadout(),
)

rag_pipeline.ingest(Observation(text="Alice likes jasmine tea.", source="dialogue"))
rag_pipeline.ingest(Observation(text="Bob prefers black coffee in the morning.", source="dialogue"))
rag_pipeline.ingest(Observation(text="Alice started learning graph-based memory systems.", source="notes"))

rag_query = Query(text="What do we know about Alice's interests?")
rag_readout = rag_pipeline.recall(rag_query)
rag_packet, _ = rag_pipeline.retrieval.run(Packet(query=rag_query), rag_pipeline.store)

print(rag_readout.text)
print("source_ids:", rag_readout.source_ids)
print()
print("retrieval trace:")
pprint(rag_packet.trace["retrieval"])

Alice likes jasmine tea.
Alice started learning graph-based memory systems.
source_ids: ['rec-1', 'rec-3']

retrieval trace:
{'candidate_count': 3,
 'embedding_candidate_count': 3,
 'module': 'embedding_similarity_retrieval',
 'reused_query_embedding': False,
 'skipped_dim_mismatch_count': 0,
 'strategy': 'embedding_similarity',
 'top_k': 2}


- 改 `top_k`
- 改写入文本
- 换 query
- 把 `EmbeddingSimilarityRetrieval` 换成 `RecencyRetrieval`

## 7. memory evolution：写入之后如何继续维护记忆

`memory_evolution` 的重点不是普通写入，而是**写入完成以后，系统是否还要额外整理、补链、重写或维护已有记忆**。

一个最直观的 graph baseline：新记录写入后，额外做邻居链接。

In [20]:
graph_topology = StoreTopology.from_layers(
    [
        StoreLayerSpec(name="default", theme="working"),
        StoreLayerSpec(
            name="knowledge_graph",
            theme="knowledge_graph",
            shape="Graph",
            indices=("graph", "entity"),
        ),
    ]
)
graph_store = MemoryStore(topology=graph_topology)

graph_pipeline = MemoryPipeline(
    representation=BasicRepresentation(elements=("text", "entities", "triple", "tags", "keywords")),
    organization=GraphAppendOrganization(target_layer="knowledge_graph"),
    evolution_trigger=ThresholdEvolutionTrigger(threshold=0.5, constant=1.0),
    memory_evolution=GraphNeighborAppendEvolution(target_layer="knowledge_graph", neighbor_limit=2),
    store=graph_store,
)

graph_a = graph_pipeline.ingest(Observation(text="Alice likes jasmine tea.", source="notes"))
graph_b = graph_pipeline.ingest(Observation(text="Alice studies graph memory systems.", source="notes"))
graph_c = graph_pipeline.ingest(Observation(text="Bob builds graph retrieval tools.", source="notes"))

print("memory evolution traces:")
pprint([
    graph_a.trace["memory_evolution"],
    graph_b.trace["memory_evolution"],
    graph_c.trace["memory_evolution"],
])
print()
print("graph layer records:")
pprint([
    {
        "record_id": record.record_id,
        "text": record.text,
        "graph": record.metadata.get("graph"),
    }
    for record in graph_store.iter_records("knowledge_graph")
])

memory evolution traces:
[{'active_unit_ids': ['unit-6abc460c37ee4f69a0b30abdded9d850'],
  'decision_source': 'evolution_decisions',
  'effects': [{'bidirectional': True,
               'candidate_count': 0,
               'candidate_record_ids': [],
               'candidate_scores': [],
               'effect_type': 'graph_link_evolution',
               'linked_record_ids': [],
               'record_id': 'rec-1',
               'rewrite_neighbor_metadata': False,
               'target_layer': 'knowledge_graph',
               'unit_id': 'unit-6abc460c37ee4f69a0b30abdded9d850'}],
  'module': 'graph_neighbor_append_evolution',
  'target_layer': 'knowledge_graph'},
 {'active_unit_ids': ['unit-cad7538b4b1f4844af7d0322e2633213'],
  'decision_source': 'evolution_decisions',
  'effects': [{'bidirectional': True,
               'candidate_count': 1,
               'candidate_record_ids': ['rec-1'],
               'candidate_scores': [{'embedding_score': 0.0,
                              

如果想更明确地看到“先触发，再演化”的链路，可以再看一个 graph-dependent 版本。

In [21]:
dependent_topology = StoreTopology.from_layers(
    [
        StoreLayerSpec(name="default", theme="working"),
        StoreLayerSpec(
            name="knowledge_graph",
            theme="knowledge_graph",
            shape="Graph",
            indices=("graph", "entity", "vector"),
        ),
    ]
)
dependent_store = MemoryStore(topology=dependent_topology)

dependent_pipeline = MemoryPipeline(
    representation=BasicRepresentation(elements=("text", "embedding", "entities", "triple", "tags", "keywords")),
    organization=GraphAppendOrganization(target_layer="knowledge_graph"),
    evolution_trigger=NeighborExistsEvolutionTrigger(target_layer="knowledge_graph", candidate_top_k=2),
    memory_evolution=(
        GraphLinkEvolution(target_layer="knowledge_graph", neighbor_limit=2, rewrite_neighbor_metadata=True),
        GraphNeighborContextTraceEvolution(target_layer="knowledge_graph", rewrite_metadata=True),
    ),
    retrieval=GraphSeedAndExpandRetrieval(top_k=4, layer="knowledge_graph", seed_top_k=1),
    readout=GraphReadout(),
    store=dependent_store,
)

dep_a = dependent_pipeline.ingest(Observation(text="Alice likes jasmine tea.", source="notes"))
dep_b = dependent_pipeline.ingest(Observation(text="Alice studies graph memory systems.", source="notes"))
dep_c = dependent_pipeline.ingest(Observation(text="Bob builds retrieval tools.", source="notes"))
dep_readout = dependent_pipeline.recall(Query(text="Alice graph"))

print("evolution trigger traces:")
pprint([
    dep_a.trace["evolution_trigger"],
    dep_b.trace["evolution_trigger"],
    dep_c.trace["evolution_trigger"],
])
print()
print("memory evolution traces:")
pprint([
    dep_a.trace["memory_evolution"],
    dep_b.trace["memory_evolution"],
    dep_c.trace["memory_evolution"],
])
print()
print(dep_readout.text)
print("source_ids:", dep_readout.source_ids)

evolution trigger traces:
[{'evolution_decisions': [False],
  'family': 'stage1_trigger_family',
  'gate': 'all',
  'module': 'neighbor_exists_evolution_trigger',
  'output_field': 'evolution_decisions',
  'per_unit': [{'decision': False,
                'gate': True,
                'score': 0.0,
                'signals': {'neighbor_count': 0.0,
                            'top_neighbor_similarity': 0.0},
                'unit_id': 'unit-94b622049f484e0e92ead91652924711'}],
  'policy': 'threshold',
  'scorer': 'identity'},
 {'evolution_decisions': [True],
  'family': 'stage1_trigger_family',
  'gate': 'all',
  'module': 'neighbor_exists_evolution_trigger',
  'output_field': 'evolution_decisions',
  'per_unit': [{'decision': True,
                'gate': True,
                'score': 1.0,
                'signals': {'neighbor_count': 1.0,
                            'top_neighbor_similarity': 0.26213046669416845},
                'unit_id': 'unit-e5982a2f523c41fa97f5ef0fd803ba45'}],


## 8. `store.check()` 是否能正常报错

`MemoryPipeline` 在构造时会做一部分即时校验，但还有一些跨模块、跨 topology 的组合约束，会注册到 `MemoryStore` 上，最后由 `store.check()` 统一检查。

下面故意构造两个错误案例：

- `EmbeddingSimilarityRetrieval` 需要 `unit.embedding`，但前面的表示层没有产出它
- `GraphAppendOrganization` 需要 graph topology，但 store 没有声明 graph layer

In [22]:
def run_store_check_demo(title, builder):
    print(f"\n=== {title} ===")
    store = MemoryStore()
    builder(store)
    print("registered modules:")
    pprint(store.registered_compositions)
    print()
    try:
        store.check()
        print("unexpected: store.check() passed")
    except IncompatibleCompositionError as exc:
        print("store.check() raised as expected:")
        print(exc)
        print("missing contracts:", store.metadata["composition_contracts"]["missing"])


run_store_check_demo(
    "missing embedding contract",
    lambda store: MemoryPipeline(
        store=store,
        representation=KeywordRepresentation(),
        retrieval=EmbeddingSimilarityRetrieval(top_k=2),
    ),
)

run_store_check_demo(
    "missing graph topology contract",
    lambda store: MemoryPipeline(
        store=store,
        organization=GraphAppendOrganization(),
    ),
)



=== missing embedding contract ===
registered modules:
({'module': 'pass_through_unit_formation',
  'produces_contracts': (),
  'requires_contracts': (),
  'slot': 'unit_formation'},
 {'module': 'keyword_representation',
  'produces_contracts': ('unit.tags',),
  'requires_contracts': (),
  'slot': 'representation'},
 {'module': 'always_write_trigger',
  'produces_contracts': (),
  'requires_contracts': (),
  'slot': 'write_trigger'},
 {'module': 'append_organization',
  'produces_contracts': (),
  'requires_contracts': (),
  'slot': 'organization'},
 {'module': 'never_evolution_trigger',
  'produces_contracts': (),
  'requires_contracts': (),
  'slot': 'evolution_trigger'},
 {'module': 'append_only_evolution',
  'produces_contracts': (),
  'requires_contracts': (),
  'slot': 'memory_evolution'},
 {'module': 'embedding_similarity_retrieval',
  'produces_contracts': (),
  'requires_contracts': ('unit.embedding',),
  'slot': 'retrieval'},
 {'module': 'concatenate_readout',
  'produces_co

如果想验证“修好之后就能通过”，可以把上面的错误组合分别改成：

- `KeywordRepresentation()` 改成 `BasicRepresentation(elements=("text", "embedding"))`
- 给 `MemoryStore` 加上带 `shape="Graph"` 的 topology

这样就能做一组很直观的“先报错，再修正，再通过”的演示。

## 9. 其他 demonstration

下面这些脚本更适合在终端里整段运行：

- `python -m memprimitive.example.demonstration.minimal_pipeline`
- `python -m memprimitive.example.demonstration.topology_store`
- `python -m memprimitive.example.demonstration.composed_triggers`
- `python -m memprimitive.example.demonstration.dispatch_organization_recall`
- `python -m memprimitive.example.demonstration.embedding_similarity_retrieval`
- `python -m memprimitive.example.demonstration.graph_baseline_pipeline`
- `python -m memprimitive.example.demonstration.graph_dependent_pipeline`
- `python -m memprimitive.example.demonstration.reflexion_reflection_cycle`
- `python -m memprimitive.example.demonstration.amem_like_graph_cycle`

其中最后两个需要真实 LLM 相关环境变量：

- `MEMPRIMITIVE_API_KEY`
- `MEMPRIMITIVE_BASE_URL`
- `MEMPRIMITIVE_MODEL`